# 🧬 Dark Manifold V25 — Generalization-First Design

## V24 Failure Analysis

V24 achieved **0.97 training correlation** but only **0.47 eval correlation** — a 0.50 gap indicating severe overfitting. Root causes:

### Problem 1: Train/Eval Distribution Mismatch
- Training: Model sees `met_target` via teacher forcing 60% of the time
- Evaluation: Model must predict autoregressively from its own outputs
- **Fix**: Scheduled sampling — gradually reduce teacher forcing during training

### Problem 2: Overparameterized Architecture
- V24 has ~8M parameters for a simple dynamics task
- SSM + Perceiver + MoE + Koopman + NeuroSymbolic + ProgramSynth + HierarchicalPlanner
- Most modules don't help — they just memorize
- **Fix**: Drastically simpler architecture focused on the core task

### Problem 3: Trivial Data Generator
- Generator uses random noise + simple growth rate
- No real metabolic coupling — S matrix barely used
- Model learns to predict noise, not biochemistry
- **Fix**: Proper ODE-based trajectory generation using stoichiometry

### Problem 4: No Validation During Training
- Training correlation measured on training data
- No early stopping, no held-out validation
- **Fix**: Proper train/val split, early stopping

---

## V25 Design Philosophy

**"Simple models that generalize beat complex models that memorize."**

V25 uses:
1. **Neural ODE** — continuous dynamics, not discrete steps
2. **Stoichiometric constraint** — physics baked in, not learned
3. **Proper validation** — held-out data, early stopping
4. **Scheduled sampling** — smooth transition from teacher forcing
5. **Minimal architecture** — ~500K params vs V24's ~8M

In [ ]:
#@title 1️⃣ Setup
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

!pip install torchdiffeq -q
from torchdiffeq import odeint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"🔥 PyTorch: {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#@title 2️⃣ Biochemistry — Realistic Stoichiometry

class Biochemistry:
    """Create biologically realistic stoichiometry matrix.
    
    Key insight: Real metabolic networks have structure:
    - Central metabolism (glycolysis, TCA) is highly connected
    - Biosynthesis pathways are linear
    - Some metabolites are hubs (ATP, NAD+, CoA)
    """
    
    def __init__(self, n_met=50, n_rxn=60, n_genes=40):
        self.n_metabolites = n_met
        self.n_reactions = n_rxn
        self.n_genes = n_genes
        
        np.random.seed(42)
        
        # Build structured stoichiometry matrix
        self.S = self._build_stoichiometry()
        self.G = self._build_gene_reaction()
        
        # Define metabolite roles
        self.hub_indices = [0, 1, 2]  # ATP, ADP, NAD+
        self.carbon_indices = list(range(3, 15))  # Central carbon
        self.biosyn_indices = list(range(15, n_met))  # Biosynthesis
        
        print(f"📊 Biochemistry created:")
        print(f"   S matrix: {self.S.shape}, density: {np.count_nonzero(self.S)/self.S.size:.1%}")
        print(f"   G matrix: {self.G.shape}")
    
    def _build_stoichiometry(self):
        """Build stoichiometry with realistic structure."""
        S = np.zeros((self.n_metabolites, self.n_reactions))
        
        # Glycolysis-like reactions (first 10 rxns)
        # Each converts one carbon metabolite to the next, consuming/producing ATP
        for j in range(10):
            # Substrate -> Product
            S[3 + j, j] = -1  # Consume substrate
            S[3 + j + 1 if j < 9 else 3, j] = 1  # Produce product (cycle)
            
            # ATP/ADP coupling (alternating)
            if j % 2 == 0:
                S[0, j] = -1  # Consume ATP
                S[1, j] = 1   # Produce ADP
            else:
                S[0, j] = 1   # Produce ATP
                S[1, j] = -1  # Consume ADP
        
        # Biosynthesis reactions (linear pathways)
        for j in range(10, self.n_reactions):
            # Pick a carbon source and biosynthesis product
            carbon = np.random.randint(3, 13)
            product = 15 + (j - 10) % (self.n_metabolites - 15)
            
            S[carbon, j] = -1
            S[product, j] = 1
            S[0, j] = -1  # Consumes ATP
            S[1, j] = 1   # Produces ADP
        
        return S
    
    def _build_gene_reaction(self):
        """Build gene-reaction associations."""
        G = np.zeros((self.n_genes, self.n_reactions))
        
        for j in range(self.n_reactions):
            # 1-2 genes per reaction
            n_genes = np.random.choice([1, 1, 2])
            genes = np.random.choice(self.n_genes, n_genes, replace=False)
            G[genes, j] = 1
        
        return G

biochem = Biochemistry(n_met=50, n_rxn=60, n_genes=40)

In [ ]:
#@title 3️⃣ ODE-Based Data Generator — The Key Fix

class MetabolicODE(nn.Module):
    """ODE dynamics using real stoichiometry.
    
    dM/dt = S @ v(M, G, condition)
    
    where v is the flux vector computed from:
    - Michaelis-Menten kinetics
    - Gene expression modulation
    - Environmental conditions
    """
    
    def __init__(self, S, G, condition_dim=4):
        super().__init__()
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        n_met, n_rxn = S.shape
        n_genes = G.shape[0]
        
        # Kinetic parameters (Vmax, Km for each reaction)
        self.vmax = nn.Parameter(torch.ones(n_rxn) * 0.1)
        self.km = nn.Parameter(torch.ones(n_rxn) * 1.0)
        
        # Condition effect on reactions
        self.condition_effect = nn.Linear(condition_dim, n_rxn)
    
    def compute_flux(self, M, G_expr, condition):
        """Compute flux vector using Michaelis-Menten kinetics."""
        n_rxn = self.S.shape[1]
        
        # Get substrate concentrations for each reaction
        # Use the first negative coefficient metabolite as substrate
        substrates = torch.zeros(M.shape[0], n_rxn, device=M.device)
        for j in range(n_rxn):
            neg_idx = (self.S[:, j] < 0).nonzero(as_tuple=True)[0]
            if len(neg_idx) > 0:
                substrates[:, j] = M[:, neg_idx[0]]
            else:
                substrates[:, j] = 1.0
        
        # Michaelis-Menten: v = Vmax * S / (Km + S)
        v = self.vmax * substrates / (self.km + substrates + 1e-6)
        
        # Gene modulation: flux scaled by enzyme expression
        # G_expr is (batch, n_genes), G is (n_genes, n_rxn)
        if G_expr.shape[1] == self.G.shape[0]:
            enzyme_level = torch.sigmoid(G_expr @ self.G)  # (batch, n_rxn)
            v = v * enzyme_level
        
        # Condition modulation
        cond_effect = torch.sigmoid(self.condition_effect(condition))
        v = v * cond_effect
        
        return v
    
    def forward(self, t, state, condition):
        """ODE right-hand side: dstate/dt"""
        n_met = self.S.shape[0]
        M = state[:, :n_met]
        G_expr = state[:, n_met:]
        
        # Compute fluxes
        v = self.compute_flux(M, G_expr, condition)
        
        # Stoichiometric dynamics: dM/dt = S @ v
        dM = v @ self.S.T
        
        # Gene expression dynamics (slow, mean-reverting)
        dG = -0.01 * (G_expr - 1.0)  # Decay toward baseline
        
        return torch.cat([dM, dG], dim=-1)


def generate_ode_trajectory(biochem, seq_len, batch_size, condition='growth', dt=0.1):
    """Generate trajectory by integrating the ODE.
    
    This is fundamentally different from V24's random walk!
    The dynamics are governed by real stoichiometry.
    """
    n_met = biochem.n_metabolites
    n_gene = biochem.n_genes
    
    # Initial conditions
    M0 = torch.exp(torch.randn(batch_size, n_met) * 0.3) + 0.5
    G0 = torch.ones(batch_size, n_gene) + torch.randn(batch_size, n_gene) * 0.1
    state0 = torch.cat([M0, G0], dim=-1)
    
    # Condition encoding
    cond = torch.zeros(batch_size, 4)
    if condition == 'growth':
        cond[:, 0] = 1.0
    elif condition == 'stress':
        cond[:, 1] = 1.0
        M0 = M0 * 0.5  # Start with lower metabolites
    elif condition == 'knockout':
        cond[:, 2] = 1.0
        # Knockout random genes
        ko_mask = torch.rand(batch_size, n_gene) < 0.15
        G0[ko_mask] = 0.01
    elif condition == 'starvation':
        cond[:, 3] = 1.0
        M0 = M0 * 0.2  # Very low starting metabolites
    
    state0 = torch.cat([M0, G0], dim=-1)
    
    # Create ODE and integrate
    ode = MetabolicODE(biochem.S, biochem.G)
    
    # Time points
    t = torch.linspace(0, seq_len * dt, seq_len)
    
    # Integrate (using simple Euler for speed, could use odeint)
    trajectory = [state0]
    state = state0
    
    for i in range(seq_len - 1):
        # Euler step with small noise for stochasticity
        dstate = ode(t[i], state, cond)
        state = state + dt * dstate + torch.randn_like(state) * 0.01
        
        # Clamp to positive
        state = torch.clamp(state, 0.01, 20.0)
        trajectory.append(state)
    
    trajectory = torch.stack(trajectory, dim=1)  # (batch, seq, state_dim)
    
    return {
        'metabolites': trajectory[:, :, :n_met],
        'genes': trajectory[:, :, n_met:],
        'condition': cond,
        'condition_name': condition
    }

# Test
test_data = generate_ode_trajectory(biochem, 30, 4, 'growth')
print(f"✅ ODE generator working")
print(f"   Metabolites: {test_data['metabolites'].shape}")
print(f"   Range: [{test_data['metabolites'].min():.2f}, {test_data['metabolites'].max():.2f}]")

In [ ]:
#@title 4️⃣ V25 Model — Simple but Principled

class DarkManifoldV25(nn.Module):
    """V25: Simplicity + Physics = Generalization
    
    Architecture:
    1. Encoder: State -> Latent (small MLP)
    2. Flux Predictor: Latent + Condition -> Flux vector
    3. Stoichiometric Update: dM = S @ flux (hard constraint)
    4. Gene Dynamics: Simple decay + condition effect
    
    Key insight: The stoichiometry matrix S is the physics.
    We only need to learn the flux function v(M, G, condition).
    """
    
    def __init__(self, n_met, n_gene, n_rxn, S, G, hidden_dim=128):
        super().__init__()
        
        self.n_met = n_met
        self.n_gene = n_gene
        self.n_rxn = n_rxn
        
        # Register stoichiometry as buffer (not learned)
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        # State encoder
        state_dim = n_met + n_gene + 4  # + condition
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Flux predictor — this is the core learned component
        self.flux_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_rxn),
        )
        
        # Gene expression update
        self.gene_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_gene),
            nn.Tanh(),  # Bounded update
        )
        
        # Learnable time constant for dynamics
        self.dt = nn.Parameter(torch.tensor(0.1))
        
    def forward(self, metabolites, genes, condition):
        """
        One step of dynamics.
        
        Returns:
            new_metabolites: Updated metabolite concentrations
            new_genes: Updated gene expression
            flux: Predicted flux vector (for analysis)
        """
        batch_size = metabolites.shape[0]
        
        # Encode state
        state = torch.cat([metabolites, genes, condition], dim=-1)
        latent = self.encoder(state)
        
        # Predict fluxes
        flux = self.flux_net(latent)
        
        # Gene modulation of flux
        if genes.shape[1] == self.G.shape[0]:
            enzyme_activity = torch.sigmoid(genes @ self.G)  # (batch, n_rxn)
            flux = flux * enzyme_activity
        
        # PHYSICS: Stoichiometric constraint
        # dM/dt = S @ v
        dM = flux @ self.S.T
        
        # Update metabolites
        dt = torch.sigmoid(self.dt) * 0.2  # Learnable but bounded
        new_metabolites = metabolites + dt * dM
        new_metabolites = torch.clamp(new_metabolites, 0.01, 20.0)
        
        # Update genes (slower dynamics)
        dG = self.gene_net(latent) * 0.1
        new_genes = genes + dG
        new_genes = torch.clamp(new_genes, 0.01, 10.0)
        
        return {
            'metabolites': new_metabolites,
            'genes': new_genes,
            'flux': flux,
            'dM': dM,
        }

# Count parameters
test_model = DarkManifoldV25(50, 40, 60, biochem.S, biochem.G)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"✅ V25 Model: {n_params:,} parameters")
print(f"   (V24 had ~8M — this is {n_params/8_000_000:.1%} of that)")

In [ ]:
#@title 5️⃣ Configuration

# Model dimensions (smaller = better generalization)
N_METABOLITES = 50
N_GENES = 40
N_REACTIONS = 60
HIDDEN_DIM = 128

# Training
N_EPOCHS = 1500
BATCH_SIZE = 32
SEQ_LEN = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3  # Strong regularization

# Scheduled sampling (key for generalization)
TEACHER_FORCING_START = 1.0  # Start with 100% teacher forcing
TEACHER_FORCING_END = 0.0    # End with 0% (full autoregressive)
TEACHER_FORCING_DECAY = 500  # Epochs to decay

# Validation
VAL_EVERY = 50
PATIENCE = 10  # Early stopping patience

print("📋 V25 Configuration:")
print(f"   Model: {N_METABOLITES} met × {N_GENES} genes × {N_REACTIONS} rxns")
print(f"   Hidden dim: {HIDDEN_DIM}")
print(f"   Epochs: {N_EPOCHS}")
print(f"   Scheduled sampling: {TEACHER_FORCING_START} → {TEACHER_FORCING_END} over {TEACHER_FORCING_DECAY} epochs")

In [ ]:
#@title 6️⃣ Initialize Model

model = DarkManifoldV25(
    n_met=N_METABOLITES,
    n_gene=N_GENES,
    n_rxn=N_REACTIONS,
    S=biochem.S,
    G=biochem.G,
    hidden_dim=HIDDEN_DIM
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-5
)

print(f"✅ Model initialized on {device}")

In [ ]:
#@title 7️⃣ Training with Scheduled Sampling + Validation

def get_teacher_forcing_ratio(epoch):
    """Linearly decay teacher forcing."""
    if epoch >= TEACHER_FORCING_DECAY:
        return TEACHER_FORCING_END
    progress = epoch / TEACHER_FORCING_DECAY
    return TEACHER_FORCING_START - progress * (TEACHER_FORCING_START - TEACHER_FORCING_END)

def evaluate(model, biochem, conditions=['growth', 'stress', 'knockout', 'starvation']):
    """Evaluate on held-out data with FULL autoregressive rollout."""
    model.eval()
    results = {}
    
    with torch.no_grad():
        for condition in conditions:
            # Generate fresh data (not seen during training)
            data = generate_ode_trajectory(biochem, SEQ_LEN, 8, condition)
            
            met = data['metabolites'][:, 0].to(device)
            gene = data['genes'][:, 0].to(device)
            cond = data['condition'].to(device)
            target = data['metabolites'].to(device)
            
            # Autoregressive rollout (no teacher forcing)
            preds = [met]
            for t in range(SEQ_LEN - 1):
                out = model(met, gene, cond)
                met = out['metabolites']
                gene = out['genes']
                preds.append(met)
            
            preds = torch.stack(preds, dim=1)
            
            # Compute correlation
            pred_flat = preds.flatten().cpu().numpy()
            target_flat = target.flatten().cpu().numpy()
            corr = np.corrcoef(pred_flat, target_flat)[0, 1]
            results[condition] = corr
    
    model.train()
    return results

# Training tracking
train_losses = []
val_corrs = []
tf_ratios = []
best_val_corr = -1
best_model_state = None
patience_counter = 0

print("🚀 Starting V25 training with scheduled sampling...")
print(f"   Epochs: {N_EPOCHS}")
print(f"   Validation every {VAL_EVERY} epochs")
print(f"   Early stopping patience: {PATIENCE}")

for epoch in tqdm(range(N_EPOCHS)):
    model.train()
    
    # Get current teacher forcing ratio
    tf_ratio = get_teacher_forcing_ratio(epoch)
    tf_ratios.append(tf_ratio)
    
    # Sample random condition
    condition = np.random.choice(['growth', 'stress', 'knockout', 'starvation'])
    
    # Generate training data
    data = generate_ode_trajectory(biochem, SEQ_LEN, BATCH_SIZE, condition)
    
    met_seq = data['metabolites'].to(device)
    gene_seq = data['genes'].to(device)
    cond = data['condition'].to(device)
    
    # Initialize
    met = met_seq[:, 0]
    gene = gene_seq[:, 0]
    
    total_loss = 0
    
    # Unroll with scheduled sampling
    for t in range(SEQ_LEN - 1):
        out = model(met, gene, cond)
        
        met_pred = out['metabolites']
        gene_pred = out['genes']
        
        # Loss
        met_target = met_seq[:, t + 1]
        gene_target = gene_seq[:, t + 1]
        
        loss = F.mse_loss(met_pred, met_target) + 0.5 * F.mse_loss(gene_pred, gene_target)
        total_loss += loss
        
        # Scheduled sampling: use target or prediction based on tf_ratio
        if torch.rand(1).item() < tf_ratio:
            met = met_target  # Teacher forcing
            gene = gene_target
        else:
            met = met_pred.detach()  # Use own prediction
            gene = gene_pred.detach()
    
    # Backprop
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    train_losses.append(total_loss.item() / SEQ_LEN)
    
    # Validation
    if epoch % VAL_EVERY == 0:
        val_results = evaluate(model, biochem)
        avg_corr = np.mean(list(val_results.values()))
        val_corrs.append((epoch, avg_corr, val_results))
        
        print(f"\nEpoch {epoch}: Loss={train_losses[-1]:.4f}, TF={tf_ratio:.2f}")
        print(f"   Val: growth={val_results['growth']:.3f}, stress={val_results['stress']:.3f}, "
              f"ko={val_results['knockout']:.3f}, starv={val_results['starvation']:.3f}")
        print(f"   Avg: {avg_corr:.3f}")
        
        # Early stopping
        if avg_corr > best_val_corr:
            best_val_corr = avg_corr
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f"   ✨ New best!")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\n⏹️ Early stopping at epoch {epoch}")
                break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model (val corr: {best_val_corr:.3f})")

In [ ]:
#@title 8️⃣ Final Evaluation

model.eval()

final_results = {}

for condition in ['growth', 'stress', 'knockout', 'starvation']:
    # Multiple runs for stability
    corrs = []
    for _ in range(5):
        data = generate_ode_trajectory(biochem, 30, 16, condition)
        
        with torch.no_grad():
            met = data['metabolites'][:, 0].to(device)
            gene = data['genes'][:, 0].to(device)
            cond = data['condition'].to(device)
            
            preds = [met]
            for t in range(29):
                out = model(met, gene, cond)
                met = out['metabolites']
                gene = out['genes']
                preds.append(met)
            
            preds = torch.stack(preds, dim=1)
            target = data['metabolites'].to(device)
            
            pred_flat = preds.flatten().cpu().numpy()
            target_flat = target.flatten().cpu().numpy()
            corr = np.corrcoef(pred_flat, target_flat)[0, 1]
            corrs.append(corr)
    
    final_results[condition] = {
        'mean': np.mean(corrs),
        'std': np.std(corrs)
    }
    print(f"{condition:12s}: {np.mean(corrs):.3f} ± {np.std(corrs):.3f}")

avg = np.mean([r['mean'] for r in final_results.values()])
print(f"\n{'AVERAGE':12s}: {avg:.3f}")
print(f"\nTarget was 0.70, V24 achieved 0.49")

In [ ]:
#@title 9️⃣ Visualization

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Training loss
ax = axes[0, 0]
ax.plot(train_losses, 'b-', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 2. Validation correlation
ax = axes[0, 1]
epochs = [v[0] for v in val_corrs]
corrs = [v[1] for v in val_corrs]
ax.plot(epochs, corrs, 'g-o', markersize=4)
ax.axhline(0.7, color='r', linestyle='--', label='Target: 0.7')
ax.axhline(0.49, color='orange', linestyle='--', label='V24: 0.49')
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg Validation Correlation')
ax.set_title('Validation Performance')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Teacher forcing schedule
ax = axes[0, 2]
ax.plot(tf_ratios, 'purple')
ax.set_xlabel('Epoch')
ax.set_ylabel('Teacher Forcing Ratio')
ax.set_title('Scheduled Sampling')
ax.grid(True, alpha=0.3)

# 4. Per-condition results
ax = axes[1, 0]
conditions = list(final_results.keys())
means = [final_results[c]['mean'] for c in conditions]
stds = [final_results[c]['std'] for c in conditions]
colors = ['green', 'orange', 'red', 'purple']
bars = ax.bar(conditions, means, yerr=stds, color=colors, alpha=0.7, capsize=5)
ax.axhline(0.7, color='r', linestyle='--', label='Target')
ax.axhline(0.49, color='gray', linestyle='--', label='V24 avg')
ax.set_ylabel('Correlation')
ax.set_title('Multi-Condition Results (V25)')
ax.legend()
ax.set_ylim(0, 1)

# 5. Sample trajectory
ax = axes[1, 1]
with torch.no_grad():
    data = generate_ode_trajectory(biochem, 30, 1, 'growth')
    met = data['metabolites'][0, 0:1].to(device)
    gene = data['genes'][0, 0:1].to(device)
    cond = data['condition'][0:1].to(device)
    
    preds = [met.cpu()]
    for t in range(29):
        out = model(met, gene, cond)
        met = out['metabolites']
        gene = out['genes']
        preds.append(met.cpu())
    
    preds = torch.stack(preds, dim=1)[0].numpy()
    targets = data['metabolites'][0].numpy()

for i in range(5):
    ax.plot(preds[:, i], '--', label=f'Pred {i}', alpha=0.7)
    ax.plot(targets[:, i], '-', label=f'Target {i}', alpha=0.5)
ax.set_xlabel('Time step')
ax.set_ylabel('Concentration')
ax.set_title('Sample Trajectory (Growth)')
ax.legend(ncol=2, fontsize=7)

# 6. Prediction scatter
ax = axes[1, 2]
ax.scatter(targets.flatten(), preds.flatten(), alpha=0.5, s=5)
ax.plot([0, targets.max()], [0, targets.max()], 'r--')
corr = np.corrcoef(targets.flatten(), preds.flatten())[0, 1]
ax.set_xlabel('Target')
ax.set_ylabel('Predicted')
ax.set_title(f'Prediction vs Target (r={corr:.3f})')

plt.tight_layout()
plt.savefig('v25_results.png', dpi=150)
plt.show()

print("\n" + "="*60)
print("V25 FINAL SUMMARY")
print("="*60)
print(f"Avg validation correlation: {avg:.3f}")
print(f"V24 comparison: {avg:.3f} vs 0.49 ({(avg-0.49)/0.49*100:+.1f}%)")
print(f"Target: 0.70")

In [ ]:
#@title 🔟 Save Model

torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'n_metabolites': N_METABOLITES,
        'n_genes': N_GENES,
        'n_reactions': N_REACTIONS,
        'hidden_dim': HIDDEN_DIM,
    },
    'train_losses': train_losses,
    'val_corrs': val_corrs,
    'tf_ratios': tf_ratios,
    'final_results': final_results,
    'best_val_corr': best_val_corr,
    'biochem': {
        'S': biochem.S,
        'G': biochem.G,
    },
    'v25_key_changes': {
        'scheduled_sampling': True,
        'ode_data_generator': True,
        'simpler_architecture': True,
        'proper_validation': True,
        'early_stopping': True,
        'stoichiometric_constraint': True,
    }
}, 'dark_manifold_v25.pt')

print("✅ Saved to dark_manifold_v25.pt")

# 📌 V25 Summary

## Key Architectural Changes

| Component | V24 | V25 | Why |
|-----------|-----|-----|-----|
| Parameters | ~8M | ~50K | Smaller = less overfitting |
| Modules | 7 (SSM, Perceiver, MoE, etc.) | 2 (Encoder, Flux) | Simpler = generalizes |
| Physics | Learned blend | Hard S constraint | dM/dt = S @ v |
| Data generator | Random walk | ODE integration | Real dynamics |

## Key Training Changes

| Aspect | V24 | V25 | Why |
|--------|-----|-----|-----|
| Teacher forcing | Fixed 60% | 100% → 0% | Scheduled sampling |
| Validation | Post-training | Every 50 epochs | Early stopping |
| Train/eval gap | 0.50 | Should be <0.10 | Proper evaluation |

## Expected Results

| Metric | V24 | V25 Target |
|--------|-----|------------|
| Training corr | 0.97 | ~0.85 |
| Eval corr | 0.47 | >0.70 |
| Gap | 0.50 | <0.15 |

The goal is NOT maximum training correlation — it's minimum train/eval gap.